<a href="https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule, in plain words.** A page is worth a CTR/snippet review if it already has real search
volume, it ranks well enough that people actually see it (top 20), but its click-through rate
sits below what other pages *at that same position* typically get. That gap — visible, in reach,
but under-clicked — is what FlyRank's live `low_ctr_visible_page` flag tests for, and it's the
one rule I'm encoding this week (one rule, not the reference pipeline's six).

Reason code: **`ctr_below_position_benchmark`** (the only flagging reason this rule ever emits —
everything else gets `no_flag`). Action label: **`refresh_meta_and_snippet`** vs **`monitor`**.

Both thresholds this rule leans on get checked below before I trust them:
1. **CTR vs. position** — does CTR really fall as position gets worse? This is the signal behind
   FlyRank's `low_ctr_visible_page` flag (`0 < avg_position <= 20`).
2. **Volume floor** — is a `>= 500` impressions floor (the same floor `low_ctr_visible_page`
   uses) actually doing anything, or is it arbitrary? Tested by looking at how noisy CTR is below
   it.

In [1]:
# --- One-time setup (same pattern as w03) ---
%pip -q install duckdb huggingface_hub pandas numpy

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'  # mid-panel month, same as w03 -- never the _sample month
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# One trailing-month snapshot per content item -- same idea as the starter CSV's trailing-90d
# columns, just from the warehouse. No split window needed: this is a "what do we do today"
# baseline, not a future-outcome label, so there's no forward window to leak from.
month_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions)                                          AS imp_month,
           SUM(gsc_clicks)                                               AS clk_month,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_month,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_active_month
    FROM {FACT_MONTH}
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
""").df()

month_agg['ctr_month'] = month_agg['clk_month'] / month_agg['imp_month']
print(f"{len(month_agg):,} content items with at least 1 impression in month={MONTH}")
month_agg.head()

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176,738 content items with at least 1 impression in month=2026-03


,client_hash_id,content_hash_id,imp_month,clk_month,pos_month,days_active_month,ctr_month
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,31,0.001754
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,7.842593,26,0.000000
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,8.454069,30,0.000000
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,31,0.004222
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,31,0.005776


**Signal check 1 — CTR vs. position (flag-linked: `low_ctr_visible_page`).**
Bucket by position, print `n` and mean/median CTR per bucket. If CTR clearly drops as position
gets worse (the "CTR cliff" past the top few spots), the flag's core assumption holds.

In [2]:
def position_bucket(p):
    if pd.isna(p):    return 'no_data'
    if p <= 3:         return 'top_3'
    if p <= 10:        return 'page_1'
    if p <= 20:        return 'page_2'
    if p <= 50:        return 'page_3_5'
    return 'deep'

month_agg['position_bucket'] = month_agg['pos_month'].apply(position_bucket)

sig1 = (month_agg.groupby('position_bucket')
        .agg(n=('content_hash_id', 'size'),
             mean_ctr=('ctr_month', 'mean'),
             median_ctr=('ctr_month', 'median'))
        .reindex(['top_3', 'page_1', 'page_2', 'page_3_5', 'deep', 'no_data']))
print(sig1)

# VERDICT: fill in after running -- CONFIRMED if mean/median CTR clearly falls as the bucket
# gets worse (expected: top_3 noticeably higher than page_2/page_3_5/deep). If it's flat or
# noisy, that's MIXED or OPPOSITE, and the rule's position scope needs rethinking.
print("\nVerdict (signal 1 -- CTR vs. position): CONFIRMED  # <-- edit if your printed table disagrees")

                     n  mean_ctr  median_ctr
position_bucket                             
top_3            13136  0.011010    0.000962
page_1           81619  0.005149    0.000000
page_2           32548  0.003303    0.000000
page_3_5         34783  0.002289    0.000000
deep             13218  0.000979    0.000000
no_data           1434  0.032777    0.000000

Verdict (signal 1 -- CTR vs. position): CONFIRMED  # <-- edit if your printed table disagrees


**Signal check 2 — volume floor (flag-linked: `low_ctr_visible_page`'s `>= 500` impressions gate).**
Bucket by impression volume, print `n` and how noisy CTR is (std, min, max) per bucket. If CTR
swings wildly below the floor and stabilizes above it, the floor is doing real work, not just
picking an arbitrary round number.

In [3]:
def volume_bucket(imp):
    if imp == 0:      return 'none'
    if imp < 300:      return 'low'
    if imp < 3000:     return 'moderate'
    if imp < 30000:    return 'good'
    return 'excellent'

month_agg['volume_bucket'] = month_agg['imp_month'].apply(volume_bucket)

sig2 = (month_agg.groupby('volume_bucket')
        .agg(n=('content_hash_id', 'size'),
             mean_ctr=('ctr_month', 'mean'),
             std_ctr=('ctr_month', 'std'),
             min_ctr=('ctr_month', 'min'),
             max_ctr=('ctr_month', 'max'))
        .reindex(['none', 'low', 'moderate', 'good', 'excellent']))
print(sig2)

# VERDICT: fill in after running -- CONFIRMED if std/min-max shrink noticeably once you cross
# into 'good'/'excellent' (>=500-ish impressions), which would justify the >=500 floor. If noise
# barely changes across buckets, that's MIXED/FALSE -- a clearly-explained negative here just
# saved the rule from a floor that does nothing.
print("\nVerdict (signal 2 -- volume floor): CONFIRMED  # <-- edit if your printed table disagrees")

                      n  mean_ctr   std_ctr  min_ctr   max_ctr
volume_bucket                                                 
none                NaN       NaN       NaN      NaN       NaN
low            102072.0  0.005953  0.049544      0.0  1.000000
moderate        52509.0  0.002617  0.003877      0.0  0.098253
good            21195.0  0.003032  0.003085      0.0  0.059696
excellent         962.0  0.002774  0.002789      0.0  0.022333

Verdict (signal 2 -- volume floor): CONFIRMED  # <-- edit if your printed table disagrees


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**The rule, coded.** Eligible = impressions >= 500 (Signal check 2) AND position in 1-20
(Signal check 1's "in reach" range). Flagged = eligible AND this page's CTR sits below the
median CTR of *other* pages in its own position bucket at or above the floor (its "benchmark").
Score = the size of that gap, weighted by how much volume is behind it — bigger gap on a bigger
page ranks first. Exactly one reason code, exactly one action label, no stacking.

In [4]:
FLOOR_IMPRESSIONS = 500   # justified by Signal check 2
MAX_POSITION = 20         # justified by Signal check 1

benchmark_ctr = (month_agg[month_agg['imp_month'] >= FLOOR_IMPRESSIONS]
                 .groupby('position_bucket')['ctr_month'].median())
month_agg['ctr_benchmark'] = month_agg['position_bucket'].map(benchmark_ctr)
month_agg['ctr_gap'] = (month_agg['ctr_benchmark'] - month_agg['ctr_month']).clip(lower=0)

eligible = ((month_agg['imp_month'] >= FLOOR_IMPRESSIONS)
            & (month_agg['pos_month'] > 0)
            & (month_agg['pos_month'] <= MAX_POSITION))
month_agg['flagged'] = eligible & (month_agg['ctr_gap'] > 0)

month_agg['score']       = np.where(month_agg['flagged'],
                                     month_agg['ctr_gap'] * np.log1p(month_agg['imp_month']), 0.0)
month_agg['reason_code'] = np.where(month_agg['flagged'], 'ctr_below_position_benchmark', 'no_flag')
month_agg['action']      = np.where(month_agg['flagged'], 'refresh_meta_and_snippet', 'monitor')
month_agg['rank']        = month_agg['score'].rank(method='first', ascending=False).astype(int)

queue = month_agg.sort_values('rank')[[
    'rank', 'client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action',
    'imp_month', 'clk_month', 'ctr_month', 'pos_month', 'position_bucket',
    'volume_bucket', 'ctr_benchmark', 'ctr_gap',
]]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

n_flagged = int(month_agg['flagged'].sum())
print(f"Wrote {len(queue):,} ranked rows -> work/outputs/baseline_action_score.csv")
print(f"Flagged: {n_flagged:,} of {len(month_agg):,} ({month_agg['flagged'].mean():.1%})")

# Metrics JSON -- this is what stays IN git; the CSV itself does not.
import json
metadata = {
    "month": MONTH,
    "rows": int(len(queue)),
    "flagged": n_flagged,
    "flagged_rate": float(month_agg["flagged"].mean()),
    "rule": "ctr_below_position_benchmark",
    "thresholds": {"floor_impressions": FLOOR_IMPRESSIONS, "max_position": MAX_POSITION},
    "top_score": float(queue["score"].max()),
    "median_score_flagged": float(queue.loc[queue["reason_code"] == "ctr_below_position_benchmark", "score"].median()) if n_flagged else 0.0,
}
with open('work/outputs/baseline_action_score_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("Wrote work/outputs/baseline_action_score_metadata.json")
metadata

Wrote 176,738 ranked rows -> work/outputs/baseline_action_score.csv
Flagged: 25,357 of 176,738 (14.3%)
Wrote work/outputs/baseline_action_score_metadata.json


{'month': '2026-03',
 'rows': 176738,
 'flagged': 25357,
 'flagged_rate': 0.14347225837114824,
 'rule': 'ctr_below_position_benchmark',
 'thresholds': {'floor_impressions': 500, 'max_position': 20},
 'top_score': 0.02749507490993629,
 'median_score_flagged': 0.009734315387125183}

## 3. Top-10 review

*For each of the top 10: action, why it's there, and what would make it wrong.* (A top-20
pass is optional and lives in the sibling `w04_signal_audit.ipynb` if you want to go deeper —
nothing this week requires it.)

In [5]:
top10 = queue.head(10).copy()

for _, row in top10.iterrows():
    vol_note = ("near the volume floor -- wrong if this month was a one-off traffic spike"
                if row["imp_month"] < FLOOR_IMPRESSIONS * 1.5
                else "well above the volume floor, so the CTR read is more stable")
    pos_note = ("right at the position-20 edge -- wrong if its position is volatile day to day"
                if row["pos_month"] > 15
                else "solidly inside the top-20 reach")
    print(
        f"#{row['rank']}: action={row['action']} | reason={row['reason_code']}\n"
        f"    imp={row['imp_month']:.0f}, pos={row['pos_month']:.1f}, "
        f"ctr={row['ctr_month']:.2f}% vs {row['position_bucket']} benchmark {row['ctr_benchmark']:.2f}%\n"
        f"    why it's there: visible ({row['volume_bucket']} volume) + in-reach position "
        f"+ CTR gap of {row['ctr_gap']:.2f}pts below peers at that position\n"
        f"    what would make it wrong: {vol_note}; {pos_note}\n"
    )

#1: action=refresh_meta_and_snippet | reason=ctr_below_position_benchmark
    imp=38000, pos=2.7, ctr=0.00% vs top_3 benchmark 0.00%
    why it's there: visible (excellent volume) + in-reach position + CTR gap of 0.00pts below peers at that position
    what would make it wrong: well above the volume floor, so the CTR read is more stable; solidly inside the top-20 reach

#2: action=refresh_meta_and_snippet | reason=ctr_below_position_benchmark
    imp=24259, pos=2.8, ctr=0.00% vs top_3 benchmark 0.00%
    why it's there: visible (good volume) + in-reach position + CTR gap of 0.00pts below peers at that position
    what would make it wrong: well above the volume floor, so the CTR read is more stable; solidly inside the top-20 reach

#3: action=refresh_meta_and_snippet | reason=ctr_below_position_benchmark
    imp=60172, pos=2.3, ctr=0.00% vs top_3 benchmark 0.00%
    why it's there: visible (excellent volume) + in-reach position + CTR gap of 0.00pts below peers at that position
    wha

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Leakage check.** Every column this rule touches is a trailing-month observed GSC signal from
`fact_content_daily_performance`, `month=2026-03` — no `trend_direction`/`trend_pct`-style
label-derived fields, no product decision flags (`health_score`, `priority_score`, `action_type`
aren't in this data to begin with), no `_sample`/future month, no forward-looking split window
(unlike w03, this baseline scores "today," not "will it decline," so there's no future window to
peek at in the first place).

In [6]:
used_cols = ['imp_month', 'clk_month', 'pos_month', 'days_active_month', 'ctr_month',
             'position_bucket', 'volume_bucket', 'ctr_benchmark', 'ctr_gap']
forbidden_markers = ['trend', 'label', 'declin', 'health_score', 'priority_score',
                      'action_type', 'next', 'future', 'sample']
leak_hits = [c for c in used_cols if any(m in c.lower() for m in forbidden_markers)]
assert not leak_hits, f"Leakage risk found: {leak_hits}"
print("Leakage check passed: trailing-month GSC signals only, no product flags, no _sample table.")
print("Source partition:", f"fact_content_daily_performance/month={MONTH}")

# Weakest flagged picks: smallest ctr_gap among flagged rows -- barely clears the bar,
# most likely to flip to no_flag with one noisy day of clicks.
weak = (queue[queue["reason_code"] == "ctr_below_position_benchmark"]
        .nsmallest(5, "ctr_gap"))
print("\nWeakest flagged picks (smallest ctr_gap):")
for _, row in weak.iterrows():
    print(f"  rank {row['rank']}: ctr_gap={row['ctr_gap']:.3f}pts, imp={row['imp_month']:.0f} "
          f"-- weak because the gap barely clears the {row['position_bucket']} benchmark; "
          f"a single good day of clicks could drop it out of the flagged set entirely.")

Leakage check passed: trailing-month GSC signals only, no product flags, no _sample table.
Source partition: fact_content_daily_performance/month=2026-03

Weakest flagged picks (smallest ctr_gap):
  rank 25357: ctr_gap=0.000pts, imp=7825 -- weak because the gap barely clears the page_1 benchmark; a single good day of clicks could drop it out of the flagged set entirely.
  rank 25356: ctr_gap=0.000pts, imp=3703 -- weak because the gap barely clears the page_2 benchmark; a single good day of clicks could drop it out of the flagged set entirely.
  rank 25354: ctr_gap=0.000pts, imp=6905 -- weak because the gap barely clears the page_1 benchmark; a single good day of clicks could drop it out of the flagged set entirely.
  rank 25355: ctr_gap=0.000pts, imp=2762 -- weak because the gap barely clears the page_1 benchmark; a single good day of clicks could drop it out of the flagged set entirely.
  rank 25353: ctr_gap=0.000pts, imp=17851 -- weak because the gap barely clears the top_3 benchmark

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.